# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library in Python. All record sets, fields, and columns are referenced by their Croissant `@id` for robust, reproducible analysis.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD file URL.

In [ ]:
# Install `mlcroissant` if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset Croissant package using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display metadata summary
meta = dataset.metadata  # do NOT treat as a dict
print("Dataset loaded!")
print(f"Name       : {meta.name}")
print(f"Identifier : {getattr(meta, 'identifier', None)}")
print(f"Version    : {getattr(meta, 'version', None)}")
print(f"License    : {getattr(meta, 'license', None)}")
print(f"Description: {meta.description}\n")

## 2. Data Overview
Review available record sets, their fields, and IDs. All references use the `@id` provided by the Croissant schema.

Let's enumerate all `RecordSet`s and each of their fields, using the Croissant metadata interface.

In [ ]:
# List available record sets and their field @ids
print("Available record sets and fields:")
rs_ids = []
for rs in dataset.metadata.record_sets:
    print(f"\nRecordSet Name: {rs.name}")
    print(f"  @id: {rs.id}")
    rs_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name:30} (field @id: {field.id})  datatype: {field.data_type}")

## 3. Data Extraction
Extract tabular records from one or more record sets (tables).

To keep analysis focused and reproducible, we explicitly choose the main data table and its `@id` as found above. All field names used are their Croissant `@id`.

Below, we load the first available record set for illustration (update as appropriate).

In [ ]:
# Choose record sets to load (use their @id)
record_set_ids = rs_ids  # All discovered record sets

dataframes = {}
# For large datasets: limit loading, or iterate one by one
print("\nLoading tables...")
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from {rs_id} with columns:\n  {list(df.columns)}")
    else:
        print(f"No records found for {rs_id}")

# Use the first DataFrame for demonstration
main_rs_id = record_set_ids[0]
print(f"\nMain table (@id): {main_rs_id}")
print('Sample records:')
display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Standardize operations by always referring to fields by their `@id`.

- Filter records where a numeric field is greater than a threshold.
- Normalize the numeric field.
- Optionally, group by a categorical field (by its `@id`).

*If you are unsure which fields are numeric or categorical, look at the printed columns above. Update the variables accordingly with their exact field `@id`.*

In [ ]:
# Pick a numeric field (by @id) from the main table
# E.g., 'Coverage', 'MW', 'Peptide Count', or any discovered field
# Please update as appropriate for a real dataset
main_df = dataframes[main_rs_id]

sample_numeric_field = None
for col in main_df.columns:
    # Heuristic: Select common quantitative columns
    if col.lower() in ['mw', 'coverage', 'peptide_count', 'molecular_weight', 'abundance', 'normalized_abundance']:
        sample_numeric_field = col
        break
if sample_numeric_field is None:
    # Fallback: Use first numeric-like column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            sample_numeric_field = col
            break

print(f"Using numeric field '@id': {sample_numeric_field}")

threshold = main_df[sample_numeric_field].mean() if sample_numeric_field else 0
print(f"Threshold for filtering: {threshold}")

if sample_numeric_field:
    filtered_df = main_df[main_df[sample_numeric_field] > threshold].copy()
    print(f"Filtered records with {sample_numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{sample_numeric_field}_normalized"] = (
        (filtered_df[sample_numeric_field] - filtered_df[sample_numeric_field].mean()) / filtered_df[sample_numeric_field].std()
    )
    print(f"\nNormalized {sample_numeric_field} for filtered records:")
    display(filtered_df[[sample_numeric_field, f"{sample_numeric_field}_normalized"]].head())

    # Try to group by a categorical field, e.g. 'description', 'accession'
    group_field = None
    for cat_col in main_df.columns:
        if cat_col.lower() in ['description', 'accession', 'gene', 'protein']:
            group_field = cat_col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[sample_numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (average {sample_numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or field relationships. Use pandas and matplotlib for inline rendering.

Below, we plot the distribution of the selected numeric field and its normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if sample_numeric_field:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(main_df[sample_numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {sample_numeric_field}")
    plt.subplot(1,2,2)
    if f"{sample_numeric_field}_normalized" in filtered_df.columns:
        sns.histplot(filtered_df[f"{sample_numeric_field}_normalized"].dropna(), kde=True, bins=30)
        plt.title(f"{sample_numeric_field} Normalized")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field to plot.")

## 6. Conclusion
We explored the FAIR^2 dataset Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells using the Croissant metadata standard via mlcroissant. By referencing every field with its proper `@id`, we maintained robust linkage to the schema while loading, filtering, and visualizing data. This approach is reproducible and directly portable to other compatible Croissant datasets with minimal code changes.

<!-- End of notebook. For further steps, add your own transformations, visualizations, or modeling as needed. -->